# 11 · Nihai test değerlendirmesi

**Bu not defteri test kümesini kullanır. Tüm seçimler (en iyi model, hibrit, ablasyon hedefi) bundan önce doğrulama kümesinde yapılıp dondurulmuştur.** Her model test kümesinde yalnızca bir kez tahmin eder; olasılıklar saklanır, sonraki tüm analizler bunlardan hesaplanır.

- Önce GPU açın (T4). Süre yaklaşık 20-30 dk (test: 1.656 görüntü × ~40 koşu)
- Oturum kopsa yeniden çalıştırın; tahmini yapılmış koşular atlanır
- Çıktılar: `outputs/final/`

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## 1 · Seçimleri dondur ve kaydet

In [ ]:
from src import final_eval as F
import json
print(json.dumps(F.freeze(), indent=2, ensure_ascii=False))

## 2 · Test olasılıkları (her koşu bir kez)

In [ ]:
F.predict_all(include_appendix=True)

## 3 · Metrikler, güven aralıkları, kalibrasyon, eşleştirilmiş testler

In [ ]:
info = F.analyze(include_appendix=True)
print(info)

In [ ]:
import pandas as pd
pd.set_option('display.width', 250, 'display.max_columns', 40)
T = pd.read_csv(F.OUT / 'main_table_test_raw.csv')
cols = ['model','n_seeds'] + [f'{k}_mean' for k in ['accuracy','balanced_accuracy','macro_f1','qwk','mcc','roc_auc_macro','mae_grade','ece15']]
T[cols].round(4)

In [ ]:
print(open(F.OUT / 'mcnemar_test.json').read())
pd.read_csv(F.OUT / 'paired_bootstrap.csv').round(4)

## 4 · CPU gecikmesi (batch 1)

In [ ]:
L = F.cpu_latency(); L.round(2)

## 5 · Çıktıları indir
Aşağıdaki hücre `outputs/final` içeriğini tek zip yapar; Drive'a kopyalanır.

In [ ]:
import shutil
shutil.make_archive('/content/final_outputs', 'zip', str(F.OUT))
shutil.copy('/content/final_outputs.zip', str(F.OUT.parent / 'final_outputs.zip'))
print('hazır: outputs/final_outputs.zip')